# 🛰️ UAV Training — YOLO Detection (Auto-Tuned)

Tek hucre calistirin — her sey otomatik.

- 🛠️ GPU/RAM/CPU otomatik algilama + parametre maximize
- 📦 `datasets.tar.gz` → pigz + mbuffer ile max hiz extraction
- ⚡ Tekrar calistirmada tamamen skip (0 saniye)
- 🔄 Checkpoint varsa otomatik devam
- 🏆 `best.pt` → mAP skoru eklenerek Drive'a yuklenir
- 📱 Tum loglar anlik gorunur (real-time streaming)

In [ ]:
##############################################################################
# YOLO UAV Training Bootstrap — Auto-Tuned for Colab (MAX SPEED)
##############################################################################

# -- Configuration ----------------------------------------------------------
REPO_URL       = "https://github.com/CELEBI-AIA/AIA-training.git"
REPO_BRANCH    = "main"
DRIVE_DATASET  = "/content/drive/MyDrive/AIA/datasets.tar.gz"
LOCAL_CACHE    = "/content/datasets_local"
DRIVE_RUNS     = "/content/drive/MyDrive/AIA/runs"
DRIVE_UPLOAD   = "/content/drive/MyDrive/AIA"
TRAIN_SCRIPT   = "uav_training/train.py"
# ---------------------------------------------------------------------------

import subprocess, sys, os, glob, time

# Force unbuffered output -> real-time logs in Colab
os.environ['PYTHONUNBUFFERED'] = '1'

def _run(cmd, check=True, **kw):
    sys.stdout.flush(); sys.stderr.flush()
    return subprocess.run(cmd, shell=True, check=check, **kw)

def _banner(msg):
    print(f"\n{'='*60}\n  {msg}\n{'='*60}", flush=True)

# === 1. Mount Google Drive ===
_banner("1/7 -- Mounting Google Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

if not os.path.isfile(DRIVE_DATASET):
    raise FileNotFoundError(f"Dataset archive not found at {DRIVE_DATASET}.")
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs(DRIVE_UPLOAD, exist_ok=True)
print("  Done: Drive mounted", flush=True)

# === 2. Clone or Pull Repository ===
_banner("2/7 -- Setting up repository")
REPO_DIR = "/content/repo"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print('  Pulling latest...', flush=True)
    _run(f"git -C {REPO_DIR} fetch --all")
    _run(f"git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")
else:
    print(f'  Cloning {REPO_URL}...', flush=True)
    _run(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
print("  Done: Repo ready", flush=True)

# === 3. Install Requirements ===
_banner("3/7 -- Installing dependencies")
req_file = os.path.join(REPO_DIR, "requirements.txt")
if os.path.isfile(req_file):
    print('  Installing requirements.txt...', flush=True)
    _run(f"{sys.executable} -m pip install --progress-bar on -r {req_file}")
print('  Verifying ultralytics & psutil...', flush=True)
_run(f"{sys.executable} -m pip install --progress-bar on ultralytics psutil", check=False)
print("  Done: Dependencies installed", flush=True)

# === 4. Dataset Extraction (MAX SPEED) ===
_banner("4/7 -- Dataset extraction (MAX SPEED)")
print('  Installing extraction tools...', flush=True)
_run('apt-get update -qq && apt-get install -y pigz mbuffer 2>&1 | tail -3', check=False)
os.makedirs(LOCAL_CACHE, exist_ok=True)
NCPU = os.cpu_count() or 2
CACHE_MARKER = os.path.join(LOCAL_CACHE, '.done')
TAR_FAST = '--no-same-owner --no-same-permissions -b 512'
HAS_MBUFFER = os.system('which mbuffer >/dev/null 2>&1') == 0
t0 = time.time()

if os.path.isfile(CACHE_MARKER):
    existing = sum(len(f) for _,_,f in os.walk(LOCAL_CACHE))
    print(f'  \u26a1 Cache complete ({existing} files) -- skip', flush=True)
else:
    existing = sum(len(f) for _,_,f in os.walk(LOCAL_CACHE))
    if existing > 100:
        print(f'  Partial cache ({existing}) -- incremental', flush=True)
        _run(f'pigz -d -c -p {NCPU} "{DRIVE_DATASET}" '
             f'| tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} '
             f'--skip-old-files --checkpoint=10000 '
             f'--checkpoint-action=echo="  %u kontrol edildi"')
    else:
        if HAS_MBUFFER:
            print(f'  Max-speed: Drive -> mbuffer(64MB) -> pigz({NCPU}) -> tar', flush=True)
            _run(f'mbuffer -i "{DRIVE_DATASET}" -m 64M -q '
                 f'| pigz -d -c -p {NCPU} '
                 f'| tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} '
                 f'--checkpoint=10000 '
                 f'--checkpoint-action=echo="  %u cikarildi"')
        else:
            print(f'  Fast: pigz({NCPU}) -> tar', flush=True)
            _run(f'pigz -d -c -p {NCPU} "{DRIVE_DATASET}" '
                 f'| tar -xf - -C "{LOCAL_CACHE}" {TAR_FAST} '
                 f'--checkpoint=10000 '
                 f'--checkpoint-action=echo="  %u cikarildi"')
    with open(CACHE_MARKER, 'w') as f: f.write('1')

print(f'  Done in {time.time()-t0:.1f}s', flush=True)

# Symlink repo/datasets -> local cache
repo_datasets = os.path.join(REPO_DIR, 'datasets')
if os.path.islink(repo_datasets) or os.path.isdir(repo_datasets):
    _run(f'rm -rf "{repo_datasets}"')
os.symlink(LOCAL_CACHE, repo_datasets)
print(f'  Symlinked {repo_datasets} -> {LOCAL_CACHE}', flush=True)
_run('df -h /content | tail -1')

# === 5. Auto Hardware Detection + Configure ===
_banner("5/7 -- Auto-detecting hardware")
os.environ['UAV_PROJECT_DIR'] = DRIVE_RUNS
os.environ['DRIVE_UPLOAD_DIR'] = DRIVE_UPLOAD
_run(f'yolo settings runs_dir="{DRIVE_RUNS}"', check=False)
print(f'  runs_dir -> {DRIVE_RUNS}', flush=True)
print(f'  DRIVE_UPLOAD_DIR -> {DRIVE_UPLOAD}', flush=True)
_run('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader', check=False)

# === 6. Launch Training ===
_banner("6/7 -- Starting training")

def find_ckpt(d):
    c = glob.glob(os.path.join(d, '**', 'weights', 'last.pt'), recursive=True)
    return max(c, key=os.path.getmtime) if c else None

ckpt = find_ckpt(DRIVE_RUNS)
script = os.path.join(REPO_DIR, TRAIN_SCRIPT)

if not os.path.isfile(script):
    raise FileNotFoundError(f'Train script not found: {script}')

if ckpt:
    print(f'  Resuming from: {ckpt}', flush=True)
    cmd = f'{sys.executable} -u "{script}" --resume'
else:
    print('  Starting fresh training', flush=True)
    cmd = f'{sys.executable} -u "{script}"'

print(f'  Command: {cmd}', flush=True)
os.chdir(REPO_DIR)
_run(cmd)

# === 7. Post-Training Summary ===
_banner("7/7 -- Post-training summary")
best_files = glob.glob(os.path.join(DRIVE_UPLOAD, 'best_*.pt'))
if best_files:
    print('  Uploaded models:', flush=True)
    for bf in sorted(best_files):
        sz = os.path.getsize(bf) / (1024*1024)
        print(f'    -> {os.path.basename(bf)} ({sz:.1f} MB)')
else:
    print('  No renamed best.pt found in Drive')
_banner("Training complete -- outputs saved to Google Drive")